# Hidden CKD

#### Features
The following are the features used in this study in order of appearance
- Date of event: Date of the screening
- Gender: Gender of the patient (M: Male, F: Female)
- Ethnicity: The ethnicity of the participant
- Age: Age of the patient (years)
- Height (cm): Height of the participant in cm
- Weight (kg): Weight of the participant in kg
- BMI: BMI of the participant
- BMI Category: Classification of the particpant BMI according to NICE guidelines
- Systolic, Diastolic: The systolic and diastolic of the partcipants
- Pulse Pressure: 
- BP Category: Classification of the particpant BP according to NICE guidelines
- Has High BP:
- Has Diabetes
- Has Kidney Disease
- On BP Medication?
- On Diabetes Medication?
- On Cholesterol Medication?
- On Other Medication?
- Family History of Kidney Disease: Whether the patient has a family history of kidney disease (Definitely Yes, Definitely Not, Not Sure)
- uACR: Urine albumin to creatinine ratio of the participants as measured using a urine dipstick (Normal, Abnormal, High Abnormal)
- CKD Risk: A calculation of CKD risk by combining research findings

This is how CKD risk is calculated

low risk = sys<140, dia<90, uACR='Normal', Has_Diabetes=False or Family_KD=False<br>
moderate risk = sys<140, dia<90, uACR='Abnormal', Has_Diabetes=True or Family_KD=True<br>
high risk = sys>=180, dia>=120, uACR='High Abnormal', Has_Diabetes=False or Family_KD=False<br>

In [1]:
import numpy as np
import pandas as pd
from src.config import PROCESSED_DATA_DIR

2025-07-29 19:23:20.933 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: /Users/Edward/Documents/GitHub/hidden-ckd


In [2]:
filename = PROCESSED_DATA_DIR / 'hidden_ckd_processed.csv'
df = pd.read_csv(filename)
df.head()

,Date,Gender,Ethnicity,S_Ethnicity,Ethnicity_Black,DOB,Age,Age_Category,Height,Weight,...,Has_Diabetes,Has_KD,Has_HD,BP_Meds,Diabetes_Meds,Cholesterol_Meds,Other_Meds,Family_KD,uACR,CKD_Risk
0,23/10/2022,Male,Black Caribbean,Black,True,21/05/1946,76.5,>70,161.0,64.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
1,23/10/2022,Male,Black African (West Africa),Black,True,25/01/1970,52.8,41-55,163.0,78.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
2,26/08/2023,Male,Black Caribbean,Black,True,14/07/2005,18.1,<25,167.0,91.0,...,False,False,False,False,False,False,False,Definitely not,Normal,Low
3,28/04/2023,Male,Black Caribbean,Black,True,25/04/1969,54.0,41-55,168.0,87.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
4,06/11/2022,Female,Black African (West Africa),Black,True,03/11/1979,43.0,41-55,187.0,109.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate


In [3]:
# New CKD risk function
def ckd_risk(row):
    if (row['Systolic'] >= 180 or
        row['Diastolic'] >= 120 or
        row['uACR'] == 'High Abnormal'):
        return 'High'
    elif (row['Systolic'] < 140 and
          row['Diastolic'] < 90 and
          row['uACR'] == 'Normal' and
          row['Family_KD'] != 'Definitely Yes' and
          row['Has_Hpt'] == False and
          row['Has_Diabetes'] == False and
          row['Has_HD'] == False):
        return 'Low'
    else:
        return 'Moderate'
df['CKD_Risk1'] = df.apply(ckd_risk, axis=1)


# Old CKD risk function
def ckd_risk(row):
    if row['uACR'] == 'High Abnormal' and row['Has_Diabetes']:
        return 'Moderate'
    elif (row['Systolic'] < 140 and row['Diastolic'] < 90 and
          row['uACR'] == 'Normal' and not row['Has_Diabetes'] and
          row['Family_KD'] != 'Definitely Yes'):
        return 'Low'
    elif row['Systolic'] > 180 or row['Diastolic'] > 120 or row['uACR'] == 'High Abnormal':
        return 'High'
    else:
        return 'Moderate'
    
df['CKD_Risk2'] = df.apply(ckd_risk, axis=1)


# Hybrid CKD risk function
def ckd_risk(row):
    if row['uACR'] == 'High Abnormal' and row['Has_Diabetes']:
        return 'Moderate'
    elif (row['Systolic'] >= 180 or
        row['Diastolic'] >= 120 or
        row['uACR'] == 'High Abnormal'):
        return 'High'
    elif (row['Systolic'] < 140 and
          row['Diastolic'] < 90 and
          row['uACR'] == 'Normal' and
          row['Family_KD'] != 'Definitely Yes' and
          row['Has_Hpt'] == False and
          row['Has_Diabetes'] == False and
          row['Has_HD'] == False):
        return 'Low'
    else:
        return 'Moderate'
    
df['CKD_Risk3'] = df.apply(ckd_risk, axis=1)


In [4]:
df[df['CKD_Risk1'] != df['CKD_Risk2']]

,Date,Gender,Ethnicity,S_Ethnicity,Ethnicity_Black,DOB,Age,Age_Category,Height,Weight,...,BP_Meds,Diabetes_Meds,Cholesterol_Meds,Other_Meds,Family_KD,uACR,CKD_Risk,CKD_Risk1,CKD_Risk2,CKD_Risk3
21,12/08/2023,Male,Black African (West Africa),Black,True,19/11/1947,75.8,>70,168.0,67.6,...,True,False,False,False,Definitely not,Normal,Moderate,Moderate,Low,Moderate
22,23/01/2023,Male,Black African (unspecified),Black,True,15/02/1970,53.0,41-55,160.0,60.2,...,True,False,False,False,Definitely yes,Normal,Moderate,Moderate,Low,Moderate
57,10/11/2022,Female,Black African (West Africa),Black,True,20/11/1958,64.0,56-70,168.0,81.8,...,False,False,False,True,Definitely not,Normal,Moderate,Moderate,Low,Moderate
114,04/12/2022,Female,Black African (West Africa),Black,True,24/11/1959,63.1,56-70,173.0,98.4,...,True,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
131,10/11/2022,Male,Black African (West Africa),Black,True,17/10/1965,57.1,56-70,154.0,54.0,...,True,False,False,False,Definitely not,Normal,Moderate,Moderate,Low,Moderate
140,03/09/2023,Male,White British,White,False,04/02/1946,77.6,>70,175.0,71.8,...,False,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate
141,03/09/2023,Male,Black African (West Africa),Black,True,20/09/1957,66.0,56-70,154.9,67.3,...,True,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
164,05/02/2023,Male,White other,White,False,10/01/2001,22.1,<25,160.0,62.0,...,True,False,False,False,Not sure,Normal,Moderate,Moderate,Low,Moderate
179,03/09/2023,Female,Black African (West Africa),Black,True,07/02/1975,48.6,41-55,170.0,116.8,...,True,False,False,False,Definitely not,Normal,Moderate,Moderate,Low,Moderate
180,03/09/2023,Male,Black African (Central Africa),Black,True,09/08/1979,44.1,41-55,172.1,81.7,...,False,False,False,False,Not sure,Normal,Moderate,Moderate,Low,Moderate


In [5]:
df[df['CKD_Risk'] != df['CKD_Risk3']]

,Date,Gender,Ethnicity,S_Ethnicity,Ethnicity_Black,DOB,Age,Age_Category,Height,Weight,...,BP_Meds,Diabetes_Meds,Cholesterol_Meds,Other_Meds,Family_KD,uACR,CKD_Risk,CKD_Risk1,CKD_Risk2,CKD_Risk3
114,04/12/2022,Female,Black African (West Africa),Black,True,24/11/1959,63.1,56-70,173.0,98.4,...,True,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
140,03/09/2023,Male,White British,White,False,04/02/1946,77.6,>70,175.0,71.8,...,False,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate
141,03/09/2023,Male,Black African (West Africa),Black,True,20/09/1957,66.0,56-70,154.9,67.3,...,True,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
212,05/02/2023,Male,Black African (West Africa),Black,True,06/06/1985,37.7,25-40,168.6,73.7,...,False,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
240,15/01/2023,Male,Black African (West Africa),Black,True,18/09/1954,68.4,56-70,150.0,67.0,...,True,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate
270,05/02/2023,Female,Black African (West Africa),Black,True,23/09/1978,44.4,41-55,185.5,130.0,...,True,True,False,False,Definitely yes,High Abnormal,High,High,Moderate,Moderate
321,09/09/2023,Female,Black African (West Africa),Black,True,23/05/1965,58.3,56-70,160.0,83.0,...,False,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate
448,26/03/2023,Female,Mixed White/Black African,Mixed,True,03/01/1952,71.3,>70,177.0,95.1,...,False,True,True,False,Definitely yes,High Abnormal,High,High,Moderate,Moderate
471,02/09/2023,Male,Black African (West Africa),Black,True,16/12/1955,67.8,56-70,151.0,81.0,...,False,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
686,30/07/2023,Female,Black African (unspecified),Black,True,05/08/1963,60.0,56-70,156.0,74.9,...,False,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate


In [6]:
df[(df['Has_Diabetes'] == True) & (df['uACR'] == 'High Abnormal')]

,Date,Gender,Ethnicity,S_Ethnicity,Ethnicity_Black,DOB,Age,Age_Category,Height,Weight,...,BP_Meds,Diabetes_Meds,Cholesterol_Meds,Other_Meds,Family_KD,uACR,CKD_Risk,CKD_Risk1,CKD_Risk2,CKD_Risk3
114,04/12/2022,Female,Black African (West Africa),Black,True,24/11/1959,63.1,56-70,173.0,98.4,...,True,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
140,03/09/2023,Male,White British,White,False,04/02/1946,77.6,>70,175.0,71.8,...,False,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate
141,03/09/2023,Male,Black African (West Africa),Black,True,20/09/1957,66.0,56-70,154.9,67.3,...,True,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
212,05/02/2023,Male,Black African (West Africa),Black,True,06/06/1985,37.7,25-40,168.6,73.7,...,False,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
240,15/01/2023,Male,Black African (West Africa),Black,True,18/09/1954,68.4,56-70,150.0,67.0,...,True,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate
270,05/02/2023,Female,Black African (West Africa),Black,True,23/09/1978,44.4,41-55,185.5,130.0,...,True,True,False,False,Definitely yes,High Abnormal,High,High,Moderate,Moderate
321,09/09/2023,Female,Black African (West Africa),Black,True,23/05/1965,58.3,56-70,160.0,83.0,...,False,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate
448,26/03/2023,Female,Mixed White/Black African,Mixed,True,03/01/1952,71.3,>70,177.0,95.1,...,False,True,True,False,Definitely yes,High Abnormal,High,High,Moderate,Moderate
471,02/09/2023,Male,Black African (West Africa),Black,True,16/12/1955,67.8,56-70,151.0,81.0,...,False,True,False,False,Definitely not,High Abnormal,High,High,Moderate,Moderate
686,30/07/2023,Female,Black African (unspecified),Black,True,05/08/1963,60.0,56-70,156.0,74.9,...,False,True,False,False,Not sure,High Abnormal,High,High,Moderate,Moderate


It was observed that the updated CKD risk function significantly affected the models' ability to generalise.

On the surface, recall percentages were all high, however model behaviour was altered such that they could only predict between Low Risk and High Risk, with barely any predictions for Moderate Risk. Even when BP values were set to normal but diabetes or hypertension were true, the models always returned High Risk. It must be stated that in the absence of qualifying systolic, diastolic, or uACR values, it is impossible to classify a patient as High Risk. A hybrid rule system was created which merged the old rules for classifying CKD risk and the new rules (even though this approach is wrong). In the end it was discovered that the source of the errors was the SMOTEENN resampling method used, which dramatically reduced the count of the majority class (Moderate Risk), meaning an unabalanced dataset was put in and another unbalanced dataset came out. The resampling method was changed to the much simpler SMOTE and the new CKD risk classification rules (which are correct) were retained, improving model generalisability.